# 00 · Setup and smoke test

This notebook runs from VS Code on a Google Colab GPU runtime. It does three things:

1. Pulls this repository from GitHub into the Colab runtime.
2. Runs the full test suite, including exact parity with the original 25.7M model.
3. Measures training throughput of every ladder rung on the current GPU and turns it into a time and compute-unit estimate.

Nothing is saved on Colab that matters: the code comes from GitHub, and the results are printed into this notebook's outputs, which live in your local `.ipynb` file.



## 1. Check the GPU

In [1]:
import subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True,
).stdout)

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07



## 2. Get the code

Set `REPO_URL` to your GitHub repository. Re-running this cell pulls the latest commit.

In [4]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"  # <- edit
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Colab already ships torch, numpy and pytest; this only fills gaps.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

commit: 56102ba


## 3. Run the tests

The parity tests load the original model's weights into the rewritten model and require identical logits.

In [5]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "tests failed - stop here and fix before benchmarking"

....................................                                     [100%]
36 passed in 21.08s




## 4. Measure throughput

Each preset trains for a few steps on random tokens at 1024-token context with bf16 autocast. If a micro-batch does not fit in memory it is halved automatically. The result includes an MFU estimate and the hours and Colab compute units the planned pretraining token budget would need on this GPU.

In [ ]:
import importlib
import json

import glm_ladder.config
import glm_ladder.model
import glm_ladder.runtime
import glm_ladder.benchmark

# Reload in dependency order so a fresh `git pull` is fully picked up.
# (Reloading only benchmark would keep the old config/model modules in memory.)
for module in (glm_ladder.config, glm_ladder.model, glm_ladder.runtime, glm_ladder.benchmark):
    importlib.reload(module)
benchmark = glm_ladder.benchmark

receipt = benchmark.run_benchmark(
    ["s-50m", "m-100m", "l-200m"],
    sequence_length=1024, micro_batch=32, steps=10,
    gradient_checkpointing=False, use_compile=False,
)
receipt["commit"] = commit
print(json.dumps(receipt, indent=2, sort_keys=True))

KeyError: 'active_per_token_estimate'

Optional: the largest rung with gradient checkpointing, which trades speed (activations are recomputed) for a larger micro-batch.

In [ ]:
checkpointed = benchmark.run_benchmark(
    ["l-200m"], sequence_length=1024, micro_batch=64, steps=10,
    gradient_checkpointing=True, use_compile=False,
)
checkpointed["commit"] = commit
print(json.dumps(checkpointed, indent=2, sort_keys=True))

## 5. Next step

Save this notebook in VS Code; the printed receipts are stored in its outputs. Share the JSON so the compute plan in `PLAN.md` can be updated with measured numbers instead of estimates.